# What Is Version Control?
**Topic:** Git & Version Control

In [ ]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go
import plotly.express as px

import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual
from ipywidgets import IntSlider, FloatSlider, Dropdown, Button, Output, HBox, VBox, Label

from IPython.display import display, HTML, clear_output
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Explain** what version control is using the "save points" mental model
- **Describe** how a commit history lets you view, compare, and restore any previous state of a project
- **Interpret** why version control is considered a non-negotiable professional requirement in data science

> **Tip:** Start by hovering over each **commit node** in the timeline chart and reading the simulated commit message before reading the explanation below.

---
## How we got here

You have now written Python code, run it in Jupyter notebooks, managed environments, used the command line, and edited files in VS Code. All of that work lives in files on your computer. Version control is what protects those files: it records every meaningful change, lets you travel back in time to any previous state, and makes it possible for a team to work on the same files without overwriting each other. This is the moment the whole tools section comes together.

---
## Why this matters for data science

Every data science job posting lists Git as a requirement. Not as a nice-to-have: as a baseline expectation. The moment you push your first project to GitHub you have a portfolio. When a model unexpectedly degrades in production, version control is how you find the commit that introduced the problem. When a teammate makes a change that breaks your code, version control is how you recover. It is the professional infrastructure that makes all the other work sustainable.

---
## Try it yourself

In [ ]:
# Widget: An interactive commit timeline
# A horizontal scatter plot with one point per commit, positioned on a time axis
# Each point is a colored circle labeled with a short commit hash (e.g., "a3f7c1")
# Hovering over a point shows:
#   - The full commit message
#   - The author and date
#   - A summary of what files changed
# A "checkout" button next to each point shows a panel simulating the file state at that commit
# Student should notice: every commit is a complete snapshot of the project,
#   not just the changes since the last commit — you can restore any point in time

---
## What's happening?

Version control is a system that records every meaningful change to a set of files over time, so you can recall specific versions later. Git is the most widely used version control system. A **repository** (repo) is a project folder that Git tracks. A **commit** is a snapshot of the entire repository at a specific moment.

| Concept | Plain English | Technical definition |
|---------|-------------|---------------------|
| Repository | A project folder that Git is watching | A directory containing a hidden `.git/` folder |
| Commit | A save point with a message | A cryptographic snapshot of all tracked files at one moment |
| Commit hash | The commit's unique ID | A 40-character SHA-1 hash (usually shown as first 7 chars) |
| Working directory | Your files as they look right now | The actual files on disk, possibly edited since last commit |
| Staging area | Changes you have marked as ready to commit | An intermediate zone between editing and saving permanently |

### The "save points" mental model

Think of a commit like a save point in a video game. You can always return to any previous save point. You can create as many save points as you like. If you explore a risky direction and it does not work out, you return to the last save point and try something else. Unlike a video game, you can also create *parallel timelines* (branches) where you try things without affecting the main save-point chain.

Hover over the commits in the widget above and notice that each one has a complete description of what changed and why.

---
## Real-world example: A model regression caught by version control

A data science team pushes a new version of their fraud detection model to production on a Monday. By Wednesday, the precision metric has dropped 4 percentage points. Without version control they would have to guess what changed. With Git, they run `git log` and immediately see that Monday's commit changed the feature engineering step for transaction timestamps.

The chart below shows the team's model performance timeline over eight weeks, with the problem commit marked.

Notice:

- **Notice:** The performance drop is clearly correlated with the commit on week 5; without version control the team would have no way to pinpoint when the regression was introduced
- **Notice:** After identifying the commit, the team uses `git revert` to undo that specific change without losing any of the other improvements made that week
- **Notice:** The investigation takes 20 minutes with version control; without it, the team estimated it would take two to three days of debugging

> **Discussion question:** The team discovers the bug was in a Jupyter notebook. When they look at the Git diff for that commit, the notebook diff is nearly unreadable because it is stored as raw JSON. What does this tell you about how Jupyter notebooks interact with version control, and what tools exist to improve the situation?

In [ ]:
# ── Model performance timeline with a version-control-identified regression ────
weeks = list(range(1, 9))
precision = [0.841, 0.847, 0.852, 0.855, 0.813, 0.814, 0.853, 0.856]
commits = [
    "feat: add device fingerprint features",
    "fix: correct timezone handling in timestamp features",
    "feat: retrain on 6-month dataset window",
    "refactor: consolidate preprocessing pipeline",
    "bug: accidentally overwrote timestamp normalization",  # the bad commit
    "chore: update dependencies",
    "fix: revert timestamp normalization to correct logic",
    "feat: add merchant category features",
]
colors = [
    PALETTE["secondary"] if i == 4 else PALETTE["primary"]
    for i in range(len(weeks))
]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=weeks, y=precision,
    mode="lines+markers",
    line=dict(color=PALETTE["muted"], width=2),
    marker=dict(color=colors, size=12, line=dict(width=2, color="white")),
    customdata=commits,
    hovertemplate="Week %{x}<br>Precision: %{y:.1%}<br>%{customdata}<extra></extra>",
    name="Model precision",
))
fig.add_annotation(
    x=5, y=0.813,
    text="Bug commit<br>git revert used to fix",
    showarrow=True, arrowhead=2,
    ax=60, ay=-40,
    font=dict(color=PALETTE["secondary"]),
)
fig.add_hline(y=0.840, line_dash="dot", line_color=PALETTE["muted"],
              annotation_text="Baseline threshold")
layout = base_layout(
    title="Fraud Model Precision Over 8 Weeks — Regression Identified via git log",
    xaxis_title="Week",
    yaxis_title="Precision",
)
layout.update(yaxis=dict(tickformat=".0%", range=[0.79, 0.87]),
              xaxis=dict(tickmode="linear", dtick=1))
fig.update_layout(layout)
fig.show()

### Version control vs no version control

| Scenario | Without version control | With version control |
|----------|------------------------|---------------------|
| Model degrades after a code change | Days of debugging to find what changed | `git log` shows exactly which commit and what changed |
| Teammate overwrites your file | Work is lost permanently | `git diff` shows the change; `git checkout` restores it |
| You want to try a risky refactor | You make a backup folder called `project_v2` | You create a branch; main is untouched |
| A collaborator needs your latest code | You email a zip file | They run `git pull` and have everything instantly |
| Regulator asks for the model used on a specific date | You hope you still have the right version somewhere | `git checkout <commit-hash>` restores the exact state |
| You accidentally delete a critical file | The file is gone | `git checkout HEAD <filename>` restores it instantly |

---
## Key takeaway

> **Version control turns a folder of files into a complete project history; every commit is a save point you can return to, compare, share, or restore — and Git makes that history permanent.**

---
*Next up: Git Basics — the five commands you will use every single day to record your work in Git*